# MHD test problems

Three classic benchmarks of ideal MHD with `MHD2D`, each showing another side of a
magnetized flow: the rotor (torsional Alfvén waves), the magnetized blast wave (strong,
anisotropic shocks at low plasma beta) and the energy budget of the Orszag–Tang vortex.
They come from the mhd-sandbox teaching code, whose solver `MHD2D` replaces. Each runs in
seconds at the resolutions here; double the cells for the published pictures.

In [ ]:
import numpy as np
import plasma_plots
import xarray as xr

from carafe import MHD2D

plasma_plots.set_backend("plotly")
plasma_plots.set_math("latex");

## The rotor

A dense disc spins in a light gas threaded by a uniform field along $x$ (Balsara and
Spicer 1999). The spin twists the field lines, and torsional Alfvén waves carry the angular
momentum away: by $t = 0.15$ the disc has slowed and been squeezed into an oval.

In [ ]:
rotor = MHD2D.rotor(96)
rotor.run(0.15)
rotor_data = rotor.dataset()
rotor_data.density.plasma.plot.slice(x="x", y="y")

The magnetic pressure shows the waves running out along the field:

In [ ]:
rotor_data.magnetic_pressure.plasma.plot.slice(x="x", y="y")

## The magnetized blast wave

A disc of high pressure in a gas at plasma beta 0.2, the field at 45 degrees (Gardiner and
Stone 2005). The field resists compression across it, so the blast runs along the field
lines and the shocked shell is oval. The total mass and energy stay constant in the
periodic box, and the field stays divergence-free across the strong shocks.

In [ ]:
blast = MHD2D.blast_wave(96)
start = blast.totals()
blast.run(0.2)
blast_data = blast.dataset()
blast_data.pressure.plasma.plot.slice(x="x", y="y")

In [ ]:
change = {name: blast.totals()[name] - start[name] for name in ("density", "energy")}
print(f"max |div B| dx = {blast.divergence():.1e}, changes {change}")

## Energy budget of the Orszag–Tang vortex

The kinetic, magnetic and thermal energies trade places as the vortex steepens into shocks:
the shocks turn kinetic and magnetic energy into heat, while the total stays constant.

In [ ]:
vortex = MHD2D.orszag_tang(64)
times = np.linspace(0.0, 0.5, 11)
budget = []
for end in times:
    vortex.run(end - vortex.time)
    data = vortex.dataset()
    kinetic = (
        0.5
        * data.density
        * (data.velocity_x**2 + data.velocity_y**2 + data.velocity_z**2)
    )
    thermal = data.pressure / (vortex.gamma - 1.0)
    cell = (1.0 / 64) ** 2
    budget.append(
        [
            float(kinetic.sum()) * cell,
            float(data.magnetic_pressure.sum()) * cell,
            float(thermal.sum()) * cell,
        ]
    )
budget = np.array(budget)
energies = xr.DataArray(
    np.column_stack((budget, budget.sum(axis=1))).T,
    dims=("part", "time"),
    coords={
        "part": ["kinetic", "magnetic", "thermal", "total"],
        "time": ("time", times, {"long_name": "$t$"}),
    },
    attrs={"long_name": "energy"},
)
energies.plasma.plot.profiles(x="time", over="part")

In [ ]:
print(f"total energy changed by {budget[-1].sum() - budget[0].sum():.1e}")